# Tutorial 04: NVF (Normal Vector Field) Fourier Factorization

This tutorial demonstrates **Normal Vector Field (NVF)** 2D Fourier factorization in `fieldwave` for curved and oblique high-contrast grating structures (e.g., Silicon circular pillars, n=3.5).

### Contents
1. **2D Geometry & Normal Vector Field (Sobel vs SDF) Visualization**
2. **Standard DFS vs NVF (Sobel) vs Fast NVF (SDF) Comparison**
3. **Convergence Analysis across Fourier Harmonics (fto)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import fieldwave
from fieldwave.on_numpy.emsolver.nvf import get_normal_vector_field_raster, to_conv_mat_nvf_2d

## 1. Create Silicon Circular Pillar (n=3.5) & Extract Normal Vector Fields (Sobel vs SDF)

In [ ]:
# 100x100 grid High-Contrast Silicon pillar
N = 100
grid_y, grid_x = np.ogrid[:N, :N]
radius = 30
dist = np.sqrt((grid_x - N//2)**2 + (grid_y - N//2)**2)

eps_matrix = np.ones((N, N), dtype=complex)
eps_matrix[dist <= radius] = 3.5**2  # Silicon pillar (n=3.5)

# Extract Normal Vector Fields
nx_sobel, ny_sobel = get_normal_vector_field_raster(eps_matrix, method='sobel')
nx_sdf, ny_sdf = get_normal_vector_field_raster(eps_matrix, method='sdf')

# Plot comparison of Sobel vs SDF normal vector fields
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(eps_matrix.real, cmap='viridis')
axes[0].set_title('Dielectric Map (eps, n=3.5)')

axes[1].imshow(nx_sobel, cmap='coolwarm')
axes[1].set_title('Sobel Normal nx (Discrete)')

axes[2].imshow(nx_sdf, cmap='coolwarm')
axes[2].set_title('Fast NVF SDF Normal nx (Continuous)')

plt.tight_layout()
plt.show()

## 2. Compare 3 Methods: Standard DFS vs NVF (Sobel) vs Fast NVF (SDF)

In [ ]:
ucell_raster = np.expand_dims(np.sqrt(eps_matrix), axis=0)

option_common = dict(
    pol=1, n_top=1.0, n_bot=1.45,
    theta=15.0 * np.pi / 180, phi=30.0 * np.pi / 180,
    wavelength=1550.0, period=[1000, 1000], thickness=[500],
    ucell=ucell_raster
)

fto_list = [1, 2, 3, 4, 5, 6, 7, 8]
R_dfs_list, R_sobel_list, R_sdf_list = [], [], []

m_dfs = fieldwave.call_mee(backend=0, fourier_type=0, fto=[1, 1], enhanced_dfs=False, **option_common)

for fto_val in fto_list:
    # 1) Standard DFS
    m_d = fieldwave.call_mee(backend=0, fourier_type=0, fto=[fto_val, fto_val], enhanced_dfs=False, **option_common)
    R_dfs_list.append(float(m_d.conv_solve().de_ri.sum()))
    
    # 2) NVF (Sobel)
    Exx_sb, Eyy_sb, Exy_sb, Eyx_sb, Epz_sb = to_conv_mat_nvf_2d(eps_matrix, fto_val, fto_val, nx=nx_sobel, ny=ny_sobel)
    r_sb = float(m_dfs.solve_for_conv(1550.0, np.array([Exx_sb]), np.array([Eyy_sb]), np.array([Epz_sb]), np.array([Exy_sb]), np.array([Eyx_sb])).de_ri.sum())
    R_sobel_list.append(r_sb)
    
    # 3) Fast NVF (SDF)
    Exx_sd, Eyy_sd, Exy_sd, Eyx_sd, Epz_sd = to_conv_mat_nvf_2d(eps_matrix, fto_val, fto_val, nx=nx_sdf, ny=ny_sdf)
    r_sd = float(m_dfs.solve_for_conv(1550.0, np.array([Exx_sd]), np.array([Eyy_sd]), np.array([Epz_sd]), np.array([Exy_sd]), np.array([Eyx_sd])).de_ri.sum())
    R_sdf_list.append(r_sd)

# Plot comparison
plt.figure(figsize=(9, 5.5))
plt.plot(fto_list, R_dfs_list, 'o--', label='Standard DFS Reflectance', color='tab:red')
plt.plot(fto_list, R_sobel_list, 's-.', label='NVF (Sobel Gradient)', color='tab:orange')
plt.plot(fto_list, R_sdf_list, 'd-', label='Fast NVF (Distance Transform SDF)', color='tab:blue', linewidth=2.5)

plt.xlabel('Fourier Harmonics (fto)')
plt.ylabel('Reflectance (R)')
plt.title('3-Method Convergence: DFS vs NVF(Sobel) vs Fast NVF(SDF)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()